# Tokenizer training provenance

**Purpose:** Document committed tokenizer artifacts and provenance; this notebook does not train or modify tokenizers.

**Inputs (committed result/provenance files read):**
- `artifacts/tokenizers/*/provenance.json`
- `artifacts/tokenizers/*.json`
- [`configs/experiment.yaml`](../configs/experiment.yaml)

**Outputs:** Rendered artifact/provenance inventory in this notebook only; no files are written and no training is run.

**Expected runtime:** ~4 seconds (loads committed CSV/JSON/parquet files only).

**Traceability links:** [`FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md) and [`CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md).

**Execution notes:** Analysis-only notebook. It resolves `ROOT` by walking upward to `plan.md`, uses no absolute paths or machine identifiers, does not download corpora, does not retrain tokenizers, and has no hidden state. Randomness: none used. Exploratory analysis: none; cells below directly present committed result files.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'plan.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def note_missing(path):
    print(f'NOTE: optional input is missing or empty: {path}. The notebook continues without it.')

def load_csv(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    return pd.read_csv(p)

def load_parquet(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    try:
        return pd.read_parquet(p)
    except Exception as exc:
        print(f'NOTE: could not read optional parquet {path}: {exc}')
        return pd.DataFrame()

def load_json(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return {}
    return json.loads(p.read_text())

def load_text(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return ''
    return p.read_text()

def safe_read_json_path(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception as exc:
        print(f'NOTE: could not read {rel(path)}: {exc}')
        return {}

print(f'Repository root: {rel(ROOT)}')


Repository root: .


## Training provenance only

Authoritative artifacts live under [`artifacts/tokenizers/`](../artifacts/tokenizers/). This notebook is analysis-only and never retrains tokenizers.

In [2]:
tokenizer_root = ROOT / 'artifacts/tokenizers'
provenance_files = sorted(tokenizer_root.glob('*/provenance.json')) if tokenizer_root.exists() else []
flat_tokenizers = sorted(tokenizer_root.glob('*.json')) if tokenizer_root.exists() else []
print(f'Provenance files: {len(provenance_files)}')
print(f'Flat tokenizer JSON files: {len(flat_tokenizers)}')

Provenance files: 9
Flat tokenizer JSON files: 9


In [3]:
rows = []
for p in provenance_files:
    data = safe_read_json_path(p)
    rows.append({
        'tokenizer_id': p.parent.name,
        'provenance_file': rel(p),
        'algorithm': data.get('algorithm') or data.get('model') or data.get('tokenizer_algorithm'),
        'vocab_size': data.get('vocab_size') or data.get('target_vocab_size'),
        'seed': data.get('seed'),
        'training_corpus': data.get('training_corpus') or data.get('corpus'),
    })
if rows:
    display(pd.DataFrame(rows).sort_values('tokenizer_id'))
else:
    note_missing('artifacts/tokenizers/*/provenance.json')

,tokenizer_id,provenance_file,algorithm,vocab_size,seed,training_corpus
0,azure-bytebpe-16384,artifacts/tokenizers/azure-bytebpe-16384/prove...,byte_level_bpe,None,20260709,azure
1,azure-bytebpe-32768,artifacts/tokenizers/azure-bytebpe-32768/prove...,byte_level_bpe,None,20260709,azure
2,azure-bytebpe-8192,artifacts/tokenizers/azure-bytebpe-8192/proven...,byte_level_bpe,None,20260709,azure
3,general-bytebpe-16384,artifacts/tokenizers/general-bytebpe-16384/pro...,byte_level_bpe,None,20260709,general
4,general-bytebpe-32768,artifacts/tokenizers/general-bytebpe-32768/pro...,byte_level_bpe,None,20260709,general
5,general-bytebpe-8192,artifacts/tokenizers/general-bytebpe-8192/prov...,byte_level_bpe,None,20260709,general
6,mixed-bytebpe-16384,artifacts/tokenizers/mixed-bytebpe-16384/prove...,byte_level_bpe,None,20260709,mixed
7,mixed-bytebpe-32768,artifacts/tokenizers/mixed-bytebpe-32768/prove...,byte_level_bpe,None,20260709,mixed
8,mixed-bytebpe-8192,artifacts/tokenizers/mixed-bytebpe-8192/proven...,byte_level_bpe,None,20260709,mixed


In [4]:
if flat_tokenizers:
    display(pd.DataFrame({'tokenizer_json': [rel(p) for p in flat_tokenizers]}))
else:
    note_missing('artifacts/tokenizers/*.json')

,tokenizer_json
0,artifacts/tokenizers/azure-bytebpe-16384.json
1,artifacts/tokenizers/azure-bytebpe-32768.json
2,artifacts/tokenizers/azure-bytebpe-8192.json
3,artifacts/tokenizers/general-bytebpe-16384.json
4,artifacts/tokenizers/general-bytebpe-32768.json
5,artifacts/tokenizers/general-bytebpe-8192.json
6,artifacts/tokenizers/mixed-bytebpe-16384.json
7,artifacts/tokenizers/mixed-bytebpe-32768.json
8,artifacts/tokenizers/mixed-bytebpe-8192.json
